# F2LLM-v2-330M — Fine-tuning MTEB French avec Unsloth

**Objectif :** Fine-tuner `codefuse-ai/F2LLM-v2-330M` sur MIRACL-fr avec MNR Loss.  
**Stack :** Unsloth (1.8–3.3x plus rapide) + SentenceTransformers + LoRA  
**Hardware :** RTX 3070 8GB  

### Pourquoi Unsloth ?
Notre training HF Trainer estimait **34 heures**. Unsloth utilise des kernels Triton custom,  
Flash Attention optimisé, et une implémentation manuelle du backward pass pour atteindre  
**1.8x à 3.3x** la vitesse standard — soit **10 à 18 heures** sur RTX 3070.  

> ⚠️ **Note :** F2LLM n'est pas dans la liste officielle Unsloth. Il sera chargé via le fallback  
> SentenceTransformers (Qwen3 backbone supporté). Vérifier les embeddings après chargement.

## 0. Installation

In [2]:
# CELLULE 0 — à exécuter EN PREMIER avant tout import
import torch
print(f"CUDA disponible : {torch.cuda.is_available()}")
print(f"GPU             : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'AUCUN'}")
print(f"PyTorch         : {torch.__version__}")
# Doit afficher CUDA disponible : True

CUDA disponible : True
GPU             : NVIDIA GeForce RTX 3070
PyTorch         : 2.12.1+cu130


In [3]:
# Unsloth — lire le guide Windows avant d'installer
# https://unsloth.ai/docs/get-started/installing-unsloth
# PyTorch doit être installé AVANT unsloth sur Windows
#
# pip install unsloth
# pip install sentence-transformers datasets
#
# Vérification

import unsloth
import sentence_transformers
print(f"Unsloth            : {unsloth.__version__}")
print(f"SentenceTransformers: {sentence_transformers.__version__}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W0628 23:11:04.996000 15456 Lib\site-packages\torch\utils\_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0628 23:11:05.170000 15456 Lib\site-packages\torch\utils\_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


🦥 Unsloth Zoo will now patch everything to make training faster!


<string>:1: FutureWarning: torch._dynamo.config.inline_inbuilt_nn_modules is deprecated and does not do anything, inline_inbuilt_nn_modules is always True. It will be removed in a future version of PyTorch.


Unsloth            : 2026.6.9
SentenceTransformers: 5.6.0


## 1. Imports & vérifications

In [5]:
import os
import sys
import subprocess
import torch
import torch.nn.functional as F
import pandas as pd
import matplotlib.pyplot as plt

from datasets import load_dataset
from sentence_transformers import InputExample, losses, SentenceTransformerTrainingArguments
from torch.utils.data import DataLoader

# Unsloth — doit être importé EN PREMIER pour patcher transformers
from unsloth import FastSentenceTransformer

print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
    print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device  : {DEVICE}")


def get_gpu_temp():
    try:
        r = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=temperature.gpu", "--format=csv,noheader"],
            encoding="utf-8"
        )
        return r.strip() + "°C"
    except:
        return "N/A"


def get_vram():
    return f"{torch.cuda.memory_allocated() / 1e9:.2f} GB"


print(f"GPU temp : {get_gpu_temp()}")
print(f"VRAM     : {get_vram()}")

PyTorch : 2.12.1+cu130
CUDA    : True
GPU     : NVIDIA GeForce RTX 3070
VRAM    : 8.6 GB
Device  : cuda
GPU temp : 47°C
VRAM     : 0.01 GB


## 2. Chargement du modèle via FastSentenceTransformer

Unsloth patche automatiquement le modèle pour :
- Kernels Flash Attention custom (plus rapides que SDPA standard)
- Backward pass manuel optimisé (évite de stocker les activations intermédiaires)
- LoRA injecté automatiquement

F2LLM n'a pas de `modules.json` → Unsloth assigne un pooling EOS par défaut  
(compatible avec le pooling du model card officiel).

In [8]:
MODEL_NAME  = "codefuse-ai/F2LLM-v2-330M"
MAX_SEQ_LEN = 256   # 256 pour tenir dans 8GB avec batch_size élevé
LOAD_4BIT   = True  # QLoRA : ~3GB VRAM pour le modèle → plus de budget pour le batch

result = FastSentenceTransformer.from_pretrained(
    MODEL_NAME,
    max_seq_length  = MAX_SEQ_LEN,
    dtype           = None,
    load_in_4bit    = True,
    load_in_8bit    = False,
    load_in_16bit   = False,
)

# Déballer selon ce que retourne réellement la fonction
print(f"Type retourné : {type(result)}")
print(f"Longueur      : {len(result) if hasattr(result, '__len__') else 'N/A'}")
print(result)


==((====))==  Unsloth 2026.6.9: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce RTX 3070. Num GPUs = 1. Max memory: 7.999 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.12.1+cu130. CUDA: 8.6. CUDA Toolkit: 13.0. Triton: 3.7.1
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/178 [00:00<?, ?it/s]

f:\codeAI\llm\.venv\lib\site-packages\bitsandbytes\backends\cuda\ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Type retourné : <class 'sentence_transformers.sentence_transformer.model.SentenceTransformer'>
Longueur      : 3
SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}, 'message': {'method': 'forward', 'method_output_name': 'last_hidden_state', 'format': 'flat'}}, 'module_output_name': 'token_embeddings', 'max_seq_length': 256, 'do_lower_case': False, 'architecture': 'Qwen3Model'})
  (1): Pooling({'embedding_dimension': 1024, 'pooling_mode': 'lasttoken', 'include_prompt': True})
  (2): Normalize({})
)


In [9]:
# Unpacking correct — 1 seul objet
model = result  # pas de tuple à déballer

print(f"VRAM après chargement : {get_vram()}")
print(f"Tokenizer             : {model.tokenizer}")

# ── Remplacer le pooling par F2LLMPooling
import torch
import torch.nn.functional as F
from sentence_transformers import SentenceTransformer

class F2LLMPooling(torch.nn.Module):
    def __init__(self, word_embedding_dimension: int):
        super().__init__()
        self.word_embedding_dimension = word_embedding_dimension
        self.config_keys = ["word_embedding_dimension"]

    def forward(self, features: dict) -> dict:
        token_embeddings = features["token_embeddings"]
        attention_mask   = features["attention_mask"]
        B      = token_embeddings.size(0)
        device = token_embeddings.device
        eos_pos = attention_mask.sum(dim=1) - 1
        emb     = token_embeddings[torch.arange(B, device=device), eos_pos]
        emb     = F.normalize(emb.float(), p=2, dim=1)
        features["sentence_embedding"] = emb
        return features

    def get_sentence_embedding_dimension(self):
        return self.word_embedding_dimension

    def save(self, output_path):
        import json, os
        with open(os.path.join(output_path, "f2llm_pooling_config.json"), "w") as f:
            json.dump({"word_embedding_dimension": self.word_embedding_dimension}, f)

    @staticmethod
    def load(input_path):
        import json, os
        with open(os.path.join(input_path, "f2llm_pooling_config.json")) as f:
            config = json.load(f)
        return F2LLMPooling(**config)


# Remplacer le module Pooling (index 1) par F2LLMPooling
hidden_dim = model[0].get_word_embedding_dimension()
print(f"Hidden dim détecté : {hidden_dim}")  # 1024 ou 896

model[1] = F2LLMPooling(word_embedding_dimension=hidden_dim)
print(f"\nPooling remplacé ✅")
print(model)

# Vérification embeddings
emb = model.encode(["bonjour", "comment ça va ?"], convert_to_tensor=True)
print(f"\nShape  : {emb.shape}")          # (2, hidden_dim)
print(f"Normes : {emb.norm(dim=1)}")     # [1.0, 1.0]

VRAM après chargement : 0.38 GB
Tokenizer             : Qwen2Tokenizer(name_or_path='codefuse-ai/F2LLM-v2-330M', vocab_size=151643, model_max_length=256, padding_side='left', truncation_side='right', special_tokens={'eos_token': '<|im_end|>', 'pad_token': '<|endoftext|>'}, added_tokens_decoder={
	151643: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151644: AddedToken("<|im_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151645: AddedToken("<|im_end|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151646: AddedToken("<|object_ref_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151647: AddedToken("<|object_ref_end|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151648: AddedToken("<|box_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=

C:\Users\aurel\AppData\Local\Temp\ipykernel_15456\870136903.py:46: FutureWarning: The `get_word_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  hidden_dim = model[0].get_word_embedding_dimension()
f:\codeAI\llm\.venv\lib\site-packages\bitsandbytes\_ops.py:239: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
f:\codeAI\llm\.venv\lib\site-packages\bitsandbytes\_ops.py:186: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
f:\codeAI\llm\.venv\lib\site-packages\bitsandbytes\_ops.py:239: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
f:\codeAI\llm\.venv\lib\site-packages\bitsandbytes\_ops.py:186:


Shape  : torch.Size([2, 896])
Normes : tensor([1., 1.], device='cuda:0')


In [11]:
# Diagnostic — inspecter ce que reçoit F2LLMPooling en pratique
class F2LLMPoolingDebug(torch.nn.Module):
    def __init__(self, word_embedding_dimension):
        super().__init__()
        self.word_embedding_dimension = word_embedding_dimension
        self.config_keys = ["word_embedding_dimension"]

    def forward(self, features):
        token_embeddings = features["token_embeddings"]
        attention_mask   = features["attention_mask"]

        print(f"token_embeddings shape : {token_embeddings.shape}")
        print(f"attention_mask shape   : {attention_mask.shape}")
        print(f"attention_mask sum[0]  : {attention_mask[0].sum()}")
        print(f"attention_mask[0]      : {attention_mask[0]}")
        print(f"features keys          : {list(features.keys())}")

        B       = token_embeddings.size(0)
        device  = token_embeddings.device
        eos_pos = attention_mask.sum(dim=1) - 1
        print(f"eos_pos                : {eos_pos}")

        emb = token_embeddings[torch.arange(B, device=device), eos_pos]
        emb = F.normalize(emb.float(), p=2, dim=1)
        features["sentence_embedding"] = emb
        return features

    def get_sentence_embedding_dimension(self):
        return self.word_embedding_dimension

    def save(self, output_path):
        import json, os
        with open(os.path.join(output_path, "f2llm_pooling_config.json"), "w") as f:
            json.dump({"word_embedding_dimension": self.word_embedding_dimension}, f)

    @staticmethod
    def load(input_path):
        import json, os
        with open(os.path.join(input_path, "f2llm_pooling_config.json")) as f:
            config = json.load(f)
        return F2LLMPoolingDebug(**config)


# Installer la version debug
model[1] = F2LLMPoolingDebug(word_embedding_dimension=896)

# Encoder UN seul texte court pour lire le debug
_ = model.encode(["Bonjour"])

f:\codeAI\llm\.venv\lib\site-packages\bitsandbytes\_ops.py:239: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
f:\codeAI\llm\.venv\lib\site-packages\bitsandbytes\_ops.py:186: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
f:\codeAI\llm\.venv\lib\site-packages\bitsandbytes\_ops.py:239: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
f:\codeAI\llm\.venv\lib\site-packages\bitsandbytes\_ops.py:186: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
f:\codeAI\llm\.venv\lib\site-packages\bitsandbytes\_

token_embeddings shape : torch.Size([1, 6, 896])
attention_mask shape   : torch.Size([1, 6])
attention_mask sum[0]  : 6
attention_mask[0]      : tensor([1, 1, 1, 1, 1, 1], device='cuda:0')
features keys          : ['input_ids', 'attention_mask', 'modality', 'token_embeddings']
eos_pos                : tensor([5], device='cuda:0')


In [14]:
# Comparer token par token entre Unsloth et HuggingFace

texts = ["Bonjour"]  # texte court, pas de padding

# ── HuggingFace direct ────────────────────────────────────────────────────────
batch = tokenizer_hf(texts, padding=True, truncation=True, return_tensors="pt").to("cuda")
print(f"HF input_ids    : {batch['input_ids']}")
print(f"HF attention    : {batch['attention_mask']}")

with torch.no_grad():
    out = hf_model(**batch)

eos_pos = batch["attention_mask"].sum(dim=1) - 1
emb_hf  = out.last_hidden_state[0, eos_pos[0]]
emb_hf  = F.normalize(emb_hf.float(), p=2, dim=0)
print(f"HF eos_pos      : {eos_pos[0].item()}")
print(f"HF emb[:5]      : {emb_hf[:5]}")

# ── Intercepter token_embeddings dans Unsloth ─────────────────────────────────
captured = {}

class F2LLMPoolingCapture(torch.nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.word_embedding_dimension = dim
        self.config_keys = ["word_embedding_dimension"]

    def forward(self, features):
        token_embeddings = features["token_embeddings"]
        attention_mask   = features["attention_mask"]
        B      = token_embeddings.size(0)
        device = token_embeddings.device

        # Capturer les input_ids pour comparer
        captured["input_ids"]        = features.get("input_ids")
        captured["token_embeddings"] = token_embeddings.detach().clone()
        captured["attention_mask"]   = attention_mask.detach().clone()

        eos_pos = attention_mask.sum(dim=1) - 1
        emb     = token_embeddings[torch.arange(B, device=device), eos_pos]
        emb     = F.normalize(emb.float(), p=2, dim=1)
        features["sentence_embedding"] = emb
        return features

    def get_sentence_embedding_dimension(self):
        return self.word_embedding_dimension

    def save(self, output_path):
        import json, os
        with open(os.path.join(output_path, "f2llm_pooling_config.json"), "w") as f:
            json.dump({"word_embedding_dimension": self.word_embedding_dimension}, f)

    @staticmethod
    def load(input_path):
        import json, os
        with open(os.path.join(input_path, "f2llm_pooling_config.json")) as f:
            config = json.load(f)
        return F2LLMPoolingCapture(**config)


model[1] = F2LLMPoolingCapture(896)
emb_st = model.encode(["Bonjour"], convert_to_tensor=True)

print(f"\nUnsloth input_ids : {captured['input_ids']}")
print(f"Unsloth eos emb[:5] : {captured['token_embeddings'][0, -1, :5]}")
print(f"ST emb[:5]          : {emb_st[0, :5]}")
print(f"\nHF input_ids      : {batch['input_ids']}")

# Sont-ils les mêmes tokens ?
print(f"\nMême input_ids ? {torch.equal(captured['input_ids'].cpu(), batch['input_ids'].cpu())}")

HF input_ids    : tensor([[ 81581, 151645]], device='cuda:0')
HF attention    : tensor([[1, 1]], device='cuda:0')
HF eos_pos      : 1
HF emb[:5]      : tensor([-0.0961,  0.1508,  0.0318,  0.0713,  0.0576], device='cuda:0')

Unsloth input_ids : tensor([[151644,    872,    198,  81581, 151645,    198]], device='cuda:0')
Unsloth eos emb[:5] : tensor([  3.4062,  19.0000,   2.2969, -19.5000,   3.8594], device='cuda:0',
       dtype=torch.bfloat16)
ST emb[:5]          : tensor([ 0.0342,  0.1909,  0.0231, -0.1959,  0.0388], device='cuda:0')

HF input_ids      : tensor([[ 81581, 151645]], device='cuda:0')

Même input_ids ? False


In [15]:
class F2LLMPoolingFixed(torch.nn.Module):
    """
    Cherche explicitement le token <|im_end|> (id=151645)
    comme le fait le model card F2LLM — indépendamment du template Unsloth.
    """
    EOS_TOKEN_ID = 151645  # <|im_end|> — token EOS de F2LLM/Qwen2

    def __init__(self, word_embedding_dimension: int):
        super().__init__()
        self.word_embedding_dimension = word_embedding_dimension
        self.config_keys = ["word_embedding_dimension"]

    def forward(self, features: dict) -> dict:
        token_embeddings = features["token_embeddings"]   # (B, seq, D)
        input_ids        = features["input_ids"]           # (B, seq)

        B      = token_embeddings.size(0)
        device = token_embeddings.device

        # Trouver la dernière occurrence de <|im_end|> dans chaque séquence
        is_eos  = (input_ids == self.EOS_TOKEN_ID)         # (B, seq) bool
        # last occurrence : on flip, argmax, re-flip
        eos_pos = is_eos.size(1) - 1 - is_eos.flip(dims=[1]).float().argmax(dim=1)

        emb = token_embeddings[torch.arange(B, device=device), eos_pos]
        emb = F.normalize(emb.float(), p=2, dim=1)

        features["sentence_embedding"] = emb
        return features

    def get_sentence_embedding_dimension(self):
        return self.word_embedding_dimension

    def save(self, output_path):
        import json, os
        with open(os.path.join(output_path, "f2llm_pooling_config.json"), "w") as f:
            json.dump({"word_embedding_dimension": self.word_embedding_dimension}, f)

    @staticmethod
    def load(input_path):
        import json, os
        with open(os.path.join(input_path, "f2llm_pooling_config.json")) as f:
            config = json.load(f)
        return F2LLMPoolingFixed(**config)


# Installer et vérifier
model[1] = F2LLMPoolingFixed(896)

texts = ["Bonjour", "Quelle est la capitale de la France ?"]
emb_st = model.encode(texts, convert_to_tensor=True)

# Comparer avec HF
batch2 = tokenizer_hf(texts, padding=True, truncation=True, return_tensors="pt").to("cuda")
with torch.no_grad():
    out2 = hf_model(**batch2)
eos2    = batch2["attention_mask"].sum(dim=1) - 1
emb_hf2 = out2.last_hidden_state[torch.arange(2), eos2]
emb_hf2 = F.normalize(emb_hf2.float(), p=2, dim=1)

cos = F.cosine_similarity(emb_hf2, emb_st.float(), dim=1)
print(f"Cosine sim : {cos}")
# Attendu : tensor([1.0000, 1.0000])

Cosine sim : tensor([0.5866, 0.7497], device='cuda:0')


In [13]:
# Voir exactement ce qu'Unsloth tokenise
print(f"Unsloth input_ids : {captured['input_ids']}")
print(f"Unsloth tokens    : {[model.tokenizer.decode([t]) for t in captured['input_ids'][0].tolist()]}")
print()
print(f"HF input_ids      : {batch['input_ids']}")
print(f"HF tokens         : {[tokenizer_hf.decode([t]) for t in batch['input_ids'][0].tolist()]}")

NameError: name 'captured' is not defined

In [ ]:
# Vérification cosine sim — doit être ~1.0
from transformers import AutoTokenizer, AutoModel

tokenizer_hf = AutoTokenizer.from_pretrained("codefuse-ai/F2LLM-v2-330M", trust_remote_code=True)
hf_model     = AutoModel.from_pretrained(
    "codefuse-ai/F2LLM-v2-330M",
    trust_remote_code=True,
    torch_dtype=torch.bfloat16
).cuda()

texts = ["Quelle est la capitale de la France ?", "Paris est une ville."]

batch = tokenizer_hf(texts, padding=True, truncation=True, return_tensors="pt").to("cuda")
with torch.no_grad():
    out = hf_model(**batch)

eos_pos = batch["attention_mask"].sum(dim=1) - 1
emb_hf  = out.last_hidden_state[torch.arange(2), eos_pos]
emb_hf  = F.normalize(emb_hf.float(), p=2, dim=1)

emb_st  = torch.tensor(model.encode(texts)).to("cuda")

cos = F.cosine_similarity(emb_hf, emb_st, dim=1)
print(f"Cosine sim : {cos}")
# Doit être tensor([1.0000, 1.0000])

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/178 [00:00<?, ?it/s]

Cosine sim : tensor([0.4437, 0.2343], device='cuda:0')


In [ ]:
# ── Vérification critique du pooling ─────────────────────────────────────────
# Unsloth assigne un pooling par défaut si pas de modules.json
# On vérifie que les embeddings sont L2-normalisés et cohérents

test_texts = [
    "Quelle est la capitale de la France ?",
    "Paris est la capitale de la France.",
    "Le bœuf bourguignon est un plat traditionnel."
]

emb = model.encode(test_texts, convert_to_tensor=True)

print(f"Shape  : {emb.shape}")          # (3, 896)
print(f"Normes : {emb.norm(dim=1)}")    # doit être ~[1.0, 1.0, 1.0]
print(f"dtype  : {emb.dtype}")

sim_pos = (emb[0] @ emb[1]).item()
sim_neg = (emb[0] @ emb[2]).item()
print(f"\nSim query–positif : {sim_pos:.4f}")
print(f"Sim query–négatif : {sim_neg:.4f}")
print(f"Margin            : {sim_pos - sim_neg:.4f}")

assert emb.shape[1] == 896, f"Dimension inattendue : {emb.shape[1]} (attendu 896)"
print("\n✅ Pooling Unsloth correct — embeddings 896D L2-normalisés")

## 3. Dataset — MIRACL French (complet)

In [ ]:
ds = load_dataset(
    "sentence-transformers/miracl",
    "fr-triplet-all",
    split="train"
)

print(f"Exemples  : {len(ds)}")
print(f"Colonnes  : {ds.column_names}")
print()
print("anchor  :", ds[0]["anchor"][:100])
print("positive:", ds[0]["positive"][:100])

## 4. Conversion en InputExample pour SentenceTransformers

Unsloth utilise l'API SentenceTransformers standard.  
On passe des paires (anchor, positive) — les négatifs sont in-batch via MNR.

In [ ]:
train_examples = [
    InputExample(texts=[row["anchor"], row["positive"]])
    for row in ds
]

print(f"Exemples MNR : {len(train_examples)}")
print(f"Exemple[0]   : {train_examples[0].texts[0][:80]}")

## 5. Loss — CachedMultipleNegativesRankingLoss

Version optimisée de MNR spécialement pour les GPUs limités en VRAM.  
`CachedMNR` calcule les embeddings en mini-chunks, puis calcule la loss sur le batch complet.  
Résultat : **batch effectif de 512+** avec seulement 4–8 séquences en VRAM à la fois.

C'est l'équivalent de GradCache mais intégré nativement dans SentenceTransformers.

In [ ]:
# DataLoader — batch_size élevé possible grâce à CachedMNR
# Unsloth + CachedMNR gèrent la VRAM en interne
BATCH_SIZE = 64   # bien plus grand qu'avant (était 4)

train_loader = DataLoader(
    train_examples,
    batch_size  = BATCH_SIZE,
    shuffle     = True,
    num_workers = 0,    # Windows
    pin_memory  = False
)

print(f"Batch size      : {BATCH_SIZE}")
print(f"Batches / epoch : {len(train_loader)}")

# CachedMNR : mini_batch_size = nb de séquences en VRAM simultanément
# Le reste est caché sur CPU et rechargé pour le backward
train_loss = losses.CachedMultipleNegativesRankingLoss(
    model,
    mini_batch_size = 8,   # 8 séquences max en VRAM → compatible 8GB
    show_progress_bar = False
)

print(f"\nLoss   : CachedMultipleNegativesRankingLoss")
print(f"Négatifs in-batch effectifs : {BATCH_SIZE - 1} par anchor")

## 6. Entraînement — SentenceTransformers fit() via Unsloth

L'API `fit()` de SentenceTransformers gère nativement :
- Progress bar avec ETA ✅
- Logging de la loss ✅  
- Warmup + scheduler ✅
- Sauvegarde checkpoints ✅
- Compatibilité Unsloth kernels ✅

In [ ]:
NUM_EPOCHS  = 3
LR          = 2e-5
WARMUP_FRAC = 0.1
OUTPUT_PATH = "./f2llm-unsloth-lora-fr"

warmup_steps = int(len(train_loader) * NUM_EPOCHS * WARMUP_FRAC)

print(f"Epochs        : {NUM_EPOCHS}")
print(f"Batches/epoch : {len(train_loader)}")
print(f"Warmup steps  : {warmup_steps}")
print(f"LR            : {LR}")
print(f"VRAM avant    : {get_vram()}")
print(f"GPU temp      : {get_gpu_temp()}")
print()
print("🚀 Démarrage entraînement Unsloth...")

In [ ]:
model.fit(
    train_objectives   = [(train_loader, train_loss)],
    epochs             = NUM_EPOCHS,
    warmup_steps       = warmup_steps,
    optimizer_params   = {"lr": LR},
    output_path        = OUTPUT_PATH,
    save_best_model    = True,
    show_progress_bar  = True,   # ✅ progress bar native — fonctionne dans VSCode
    checkpoint_path    = "./checkpoints",
    checkpoint_save_steps = len(train_loader),  # checkpoint à chaque epoch
)

print(f"\n✅ Entraînement terminé")
print(f"VRAM après : {get_vram()}")
print(f"GPU temp   : {get_gpu_temp()}")
print(f"Modèle     : {OUTPUT_PATH}")

## 7. Sauvegarde — LoRA adapters + modèle mergé

Unsloth propose deux modes de sauvegarde :
- `save_pretrained()` → seulement les adaptateurs LoRA (~10 MB)
- `save_pretrained_merged()` → modèle complet mergé (~600 MB) prêt pour MTEB

In [ ]:
LORA_PATH   = "./f2llm-unsloth-lora-adapters"
MERGED_PATH = "./f2llm-unsloth-merged"

# Sauvegarde LoRA seul (~10 MB)
model.save_pretrained(LORA_PATH)
print(f"✅ LoRA adapters : {LORA_PATH}")

# Sauvegarde modèle mergé (~600 MB) — nécessaire pour MTEB
model.save_pretrained_merged(MERGED_PATH)
print(f"✅ Modèle mergé  : {MERGED_PATH}")

print("\nFichiers sauvegardés :")
for p in [LORA_PATH, MERGED_PATH]:
    if os.path.exists(p):
        total = sum(
            os.path.getsize(os.path.join(p, f))
            for f in os.listdir(p)
            if os.path.isfile(os.path.join(p, f))
        )
        print(f"  {p:<40} {total/1e6:.0f} MB")

## 8. Vérification qualitative post-training

In [ ]:
# Charger le modèle mergé pour l'évaluation
model_eval = FastSentenceTransformer.from_pretrained(
    MERGED_PATH,
    for_inference = True,   # ⚠️ OBLIGATOIRE pour l'inférence avec Unsloth
    max_seq_length = MAX_SEQ_LEN,
    dtype = None,
)

pairs = [
    (
        "Quelle est la capitale de la France ?",
        "Paris est la capitale et la plus grande ville de France.",
        "Le bœuf bourguignon est un plat traditionnel français."
    ),
    (
        "Comment fonctionne un moteur électrique ?",
        "Un moteur électrique convertit l'énergie électrique en énergie mécanique.",
        "Les voitures thermiques utilisent de l'essence ou du diesel."
    ),
    (
        "Quels sont les symptômes du diabète ?",
        "Les symptômes incluent une soif excessive, une fatigue et des urinations fréquentes.",
        "Le Tour de France est une course cycliste annuelle."
    ),
    (
        "Qui a écrit Les Misérables ?",
        "Les Misérables est un roman de Victor Hugo publié en 1862.",
        "La recette de la tarte tatin utilise des pommes caramélisées."
    ),
]

print(f"{'Query':<45} {'sim(+)':>7} {'sim(-)':>7} {'margin':>8}")
print("-" * 75)

for query, pos, neg in pairs:
    eq = model_eval.encode([query], convert_to_tensor=True)
    ep = model_eval.encode([pos],   convert_to_tensor=True)
    en = model_eval.encode([neg],   convert_to_tensor=True)

    sim_pos = (eq @ ep.T).item()
    sim_neg = (eq @ en.T).item()
    margin  = sim_pos - sim_neg

    ok = "✅" if margin > 0 else "❌"
    print(f"{ok} {query[:43]:<45} {sim_pos:>7.4f} {sim_neg:>7.4f} {margin:>8.4f}")

## 9. Comparaison baseline vs fine-tuné (même pairs)

In [ ]:
# Charger le modèle baseline (sans fine-tuning)
model_base = FastSentenceTransformer.from_pretrained(
    MODEL_NAME,
    for_inference  = True,
    max_seq_length = MAX_SEQ_LEN,
    load_in_4bit   = LOAD_4BIT,
)

results = []
for query, pos, neg in pairs:
    # Baseline
    eq_b = model_base.encode([query], convert_to_tensor=True)
    ep_b = model_base.encode([pos],   convert_to_tensor=True)
    en_b = model_base.encode([neg],   convert_to_tensor=True)
    margin_base = (eq_b @ ep_b.T).item() - (eq_b @ en_b.T).item()

    # Fine-tuné
    eq_f = model_eval.encode([query], convert_to_tensor=True)
    ep_f = model_eval.encode([pos],   convert_to_tensor=True)
    en_f = model_eval.encode([neg],   convert_to_tensor=True)
    margin_ft = (eq_f @ ep_f.T).item() - (eq_f @ en_f.T).item()

    results.append({
        "query"        : query[:40],
        "margin_base"  : round(margin_base, 4),
        "margin_ft"    : round(margin_ft,   4),
        "delta"        : round(margin_ft - margin_base, 4)
    })

df = pd.DataFrame(results)
print(df.to_string(index=False))
print(f"\nMarge moyenne baseline  : {df['margin_base'].mean():.4f}")
print(f"Marge moyenne fine-tuné : {df['margin_ft'].mean():.4f}")
print(f"Gain moyen              : {df['delta'].mean():.4f}")

## 10. Évaluation MTEB French

`pip install mteb` si pas installé.

In [ ]:
import mteb
from sentence_transformers import SentenceTransformer

# Wrapper MTEB — attend model.encode(sentences) -> np.ndarray
class MTEBWrapper:
    def __init__(self, st_model):
        self.st_model = st_model

    def encode(self, sentences, batch_size=32, **kwargs):
        return self.st_model.encode(
            sentences,
            batch_size         = batch_size,
            convert_to_numpy   = True,
            normalize_embeddings = True
        )


# Tâches French représentatives (subset rapide ~30 min)
TASKS_FR = [
    "MintakaRetrieval",
    "MLSUMClusteringS2S",
    "MasakhaNEWSClassification",
]

tasks = mteb.get_tasks(tasks=TASKS_FR, languages=["fra"])

# ── Fine-tuné ────────────────────────────────────────────────────────────────
mteb_ft   = MTEBWrapper(model_eval)
eval_ft   = mteb.MTEB(tasks=tasks)
results_ft = eval_ft.run(mteb_ft, output_folder="./mteb_results_unsloth_ft", verbosity=1)

# ── Baseline ─────────────────────────────────────────────────────────────────
mteb_base   = MTEBWrapper(model_base)
eval_base   = mteb.MTEB(tasks=tasks)
results_base = eval_base.run(mteb_base, output_folder="./mteb_results_unsloth_base", verbosity=1)

# ── Tableau comparatif ────────────────────────────────────────────────────────
base_scores = {r.task_name: r.get_score() for r in results_base}
ft_scores   = {r.task_name: r.get_score() for r in results_ft}

rows = []
for task_name in ft_scores:
    base = base_scores.get(task_name, float("nan"))
    ft   = ft_scores[task_name]
    rows.append({
        "task"     : task_name,
        "baseline" : round(base, 4),
        "finetuned": round(ft,   4),
        "delta"    : round(ft - base, 4)
    })

df_mteb = pd.DataFrame(rows)
print("\n─── Résultats MTEB French ───")
print(df_mteb.to_string(index=False))
print(f"\nDelta moyen : {df_mteb['delta'].mean():.4f}")

## 11. Rechargement depuis les adapters LoRA (optionnel)

Pour éviter de sauvegarder 600 MB, on peut merger à la demande.

In [ ]:
# Recharger depuis les adapters LoRA uniquement
model_from_lora = FastSentenceTransformer.from_pretrained(
    LORA_PATH,
    for_inference  = True,
    max_seq_length = MAX_SEQ_LEN,
)

# Vérification cohérence
emb_merged = model_eval.encode(["bonjour"], convert_to_tensor=True)
emb_lora   = model_from_lora.encode(["bonjour"], convert_to_tensor=True)

cos = F.cosine_similarity(emb_merged.float(), emb_lora.float(), dim=1)
print(f"Cosine sim LoRA vs merged : {cos.item():.6f}")
# Doit être ~1.0000